# TP2 — Entrenamiento y optimización

Comparación de modelos y control de sobreajuste. **La configuración final es provisional:** no usar Test para ajustar hiperparámetros.

In [ ]:
import pandas as pd
from pathlib import Path

# Datos procesados
ruta = Path("/content/drive/MyDrive/DiploIA/TP2/1-data/2-processed")

# Datos de entrenamiento
X_train = pd.read_csv(ruta / "X_train.csv")
y_train = pd.read_csv(ruta / "y_train.csv").squeeze("columns")

# Dimensiones
print("Train:", X_train.shape)
print("Objetivo:", y_train.shape)

print("\nDistribución de clases:")
print(y_train.value_counts(normalize=True).round(4))

## División de Train en entrenamiento y validación

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

X_entrenamiento, X_validacion, y_entrenamiento, y_validacion = train_test_split(
    X_train, y_train, test_size=0.3, random_state=42, stratify=y_train
)

modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

for nombre, modelo in modelos.items():
    modelo.fit(X_entrenamiento, y_entrenamiento)
    print(f"\nModelo: {nombre}")
    print(classification_report(y_validacion, modelo.predict(X_validacion)))

Evalué 3 modelos de clasificaciopn con el objetivo de encontrar auql tiene mejor acuracy, mediante una división del conjunto de entrenamiento en entrenamiento (70 %) y validación (30 %) para comparar elos y evaluar overfitting. Random Forest obtuvo el mejor desempeño, con un F1-score de 0.71 para la clase fumador y una exactitud del 78 %.

In [ ]:
from sklearn.metrics import f1_score

# Comparar rendimiento en Train y Validación
for nombre, modelo in modelos.items():

    # Entrenar el modelo
    modelo.fit(X_entrenamiento, y_entrenamiento)

    # Predicciones
    pred_train = modelo.predict(X_entrenamiento)
    pred_val = modelo.predict(X_validacion)

    # Calcular F1-score
    f1_train = f1_score(y_entrenamiento, pred_train)
    f1_val = f1_score(y_validacion, pred_val)

    # Diferencia entre ambos resultados
    diferencia = f1_train - f1_val

    print(f"\nModelo: {nombre}")
    print(f"F1 Train: {f1_train:.4f}")
    print(f"F1 Validación: {f1_val:.4f}")
    print(f"Diferencia: {diferencia:.4f}")

## Regularización del árbol de decisión

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# Probar diferentes profundidades
profundidades = [3, 5, 8, 10, 15, 20]

for profundidad in profundidades:

    modelo_arbol = DecisionTreeClassifier(
        max_depth=profundidad,
        random_state=42
    )

    modelo_arbol.fit(X_entrenamiento, y_entrenamiento)

    pred_train = modelo_arbol.predict(X_entrenamiento)
    pred_val = modelo_arbol.predict(X_validacion)

    f1_train = f1_score(y_entrenamiento, pred_train)
    f1_val = f1_score(y_validacion, pred_val)

    diferencia = f1_train - f1_val

    print(
        f"Profundidad: {profundidad} | "
        f"Train: {f1_train:.4f} | "
        f"Validación: {f1_val:.4f} | "
        f"Diferencia: {diferencia:.4f}"
    )

Evaluación del sobreajuste según la profundidad máxima del Árbol de Decisión

In [ ]:
import matplotlib.pyplot as plt

# Resultados del Árbol de Decisión
profundidades = [3, 5, 8, 10, 15, 20]

f1_train = [0.6348, 0.6859, 0.6904, 0.7197, 0.8644, 0.9637]
f1_validacion = [0.6231, 0.6720, 0.6513, 0.6263, 0.6431, 0.6306]

# Graficar resultados
plt.figure(figsize=(9, 5))

plt.plot(profundidades, f1_train, marker="o", label="Entrenamiento")
plt.plot(profundidades, f1_validacion, marker="o", label="Validación")

plt.axvline(x=5, color="gray", linestyle="--", label="Profundidad seleccionada")

plt.xlabel("Profundidad máxima del árbol")
plt.ylabel("F1-score (fumadores)")
plt.title("Árbol de Decisión: F1-score según profundidad")

plt.xticks(profundidades)
plt.ylim(0.55, 1.0)
plt.legend()
plt.grid(True)

plt.show()

## Regularización de Random Forest

# Evaluar la profundidad máxima de Random Forest

In [ ]:
# Evaluar diferentes profundidades máximas en Random Forest

profundidades = [3, 5, 8, 10, 15, 20]

resultados_rf_train = []
resultados_rf_validacion = []

for profundidad in profundidades:

    modelo_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=profundidad,
        random_state=42,
        n_jobs=-1
    )

    modelo_rf.fit(X_entrenamiento, y_entrenamiento)

    pred_train = modelo_rf.predict(X_entrenamiento)
    pred_val = modelo_rf.predict(X_validacion)

    f1_train = f1_score(y_entrenamiento, pred_train)
    f1_val = f1_score(y_validacion, pred_val)

    resultados_rf_train.append(f1_train)
    resultados_rf_validacion.append(f1_val)

    diferencia = f1_train - f1_val

    print(
        f"Profundidad: {profundidad} | "
        f"Train: {f1_train:.4f} | "
        f"Validación: {f1_val:.4f} | "
        f"Diferencia: {diferencia:.4f}"
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    profundidades,
    resultados_rf_train,
    marker="o",
    label="Entrenamiento"
)

plt.plot(
    profundidades,
    resultados_rf_validacion,
    marker="o",
    label="Validación"
)

plt.axvline(
    x=10,
    color="gray",
    linestyle="--",
    label="Profundidad a explorar"
)

plt.xlabel("Profundidad máxima de los árboles")
plt.ylabel("F1-score (fumadores)")
plt.title("Random Forest: F1-score según profundidad máxima")

plt.xticks(profundidades)
plt.ylim(0.55, 1.0)
plt.legend()
plt.grid(True)

plt.show()

Evaluar min_samples_leaf en Random Forest

In [ ]:
# Evaluar min_samples_leaf en Random Forest

valores_leaf = [1, 2, 5, 10, 20]
resultados_leaf_train = []
resultados_leaf_validacion = []

for leaf in valores_leaf:
    modelo_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=leaf,
        random_state=42,
        n_jobs=-1)
    modelo_rf.fit(X_entrenamiento, y_entrenamiento)
    pred_train = modelo_rf.predict(X_entrenamiento)
    pred_val = modelo_rf.predict(X_validacion)
    f1_train = f1_score(y_entrenamiento, pred_train)
    f1_val = f1_score(y_validacion, pred_val)

    resultados_leaf_train.append(f1_train)
    resultados_leaf_validacion.append(f1_val)

    diferencia = f1_train - f1_val

    print(f"min_samples_leaf: {leaf} | "
        f"Train: {f1_train:.4f} | "
        f"Validación: {f1_val:.4f} | "
        f"Diferencia: {diferencia:.4f}")

In [ ]:

plt.figure(figsize=(9, 5))

plt.plot(
    valores_leaf,
    resultados_leaf_train,
    marker="o",
    label="Entrenamiento")

plt.plot(
    valores_leaf,
    resultados_leaf_validacion,
    marker="o",
    label="Validación")

plt.xlabel("Cantidad mínima de observaciones por hoja")
plt.ylabel("F1-score (fumadores)")
plt.title("Random Forest: F1-score según min_samples_leaf")

plt.xticks(valores_leaf)
plt.ylim(0.55, 1.0)
plt.legend()
plt.grid(True)

plt.show()

Se evaluaron diferentes valores de min_samples_leaf para analizar su efecto sobre el sobreajuste de Random Forest, manteniendo una profundidad máxima de 15. Los resultados muestran que aumentar la cantidad mínima de observaciones por hoja reduce la diferencia entre el F1-score de entrenamiento y validación.

Con min_samples_leaf=10, la diferencia se redujo de 0.2284 a 0.1309, mientras que el F1 de validación disminuyó de 0.7060 a 0.6961. Esto evidencia que es posible controlar parcialmente el sobreajuste sin perder demasiado rendimiento predictivo.

Se continuará evaluando otras configuraciones para encontrar un mejor equilibrio entre rendimiento y capacidad de generalización.

# Evaluar combinaciones de parámetros en Random Forest


In [ ]:

profundidades = [8, 10, 15, 20]
valores_leaf = [2, 5, 10, 20]

resultados_combinaciones = []

for profundidad in profundidades:
    for leaf in valores_leaf:

        modelo_rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=profundidad,
            min_samples_leaf=leaf,
            random_state=42,
            n_jobs=-1
        )

        modelo_rf.fit(X_entrenamiento, y_entrenamiento)

        pred_train = modelo_rf.predict(X_entrenamiento)
        pred_val = modelo_rf.predict(X_validacion)

        f1_train = f1_score(y_entrenamiento, pred_train)
        f1_val = f1_score(y_validacion, pred_val)

        diferencia = f1_train - f1_val

        resultados_combinaciones.append({
            "max_depth": profundidad,
            "min_samples_leaf": leaf,
            "F1 Train": f1_train,
            "F1 Validación": f1_val,
            "Diferencia": diferencia
        })

# Convertir resultados en DataFrame
df_resultados_rf = pd.DataFrame(resultados_combinaciones)

# Ordenar por mayor F1 de validación
df_resultados_rf = df_resultados_rf.sort_values(
    by="F1 Validación",
    ascending=False
)

display(df_resultados_rf.round(4))

In [ ]:
# Mostrar las configuraciones con mejor F1 de validación

df_resultados_rf = df_resultados_rf.reset_index(drop=True)

display(df_resultados_rf.head(10).round(4))

La evaluación conjunta de max_depth y min_samples_leaf permitió identificar configuraciones con diferentes niveles de sobreajuste. Los modelos más complejos alcanzaron mayores valores de F1 en validación, pero también presentaron diferencias importantes respecto del entrenamiento.

La combinación max_depth=10 y min_samples_leaf=5 obtuvo un F1 de validación de 0.6927 y una diferencia de 0.0728, representando una alternativa con menor sobreajuste que las configuraciones de mayor profundidad.

Se continuará evaluando el modelo antes de seleccionar los parámetros definitivos.

# Evaluar min_samples_split en Random Forest

In [ ]:
valores_split = [2, 5, 10, 20, 50]

resultados_split_train = []
resultados_split_validacion = []

for split in valores_split:

    modelo_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=5,
        min_samples_split=split,
        random_state=42,
        n_jobs=-1)

    modelo_rf.fit(X_entrenamiento, y_entrenamiento)

    pred_train = modelo_rf.predict(X_entrenamiento)
    pred_val = modelo_rf.predict(X_validacion)

    f1_train = f1_score(y_entrenamiento, pred_train)
    f1_val = f1_score(y_validacion, pred_val)

    resultados_split_train.append(f1_train)
    resultados_split_validacion.append(f1_val)

    diferencia = f1_train - f1_val

    print(
        f"min_samples_split: {split} | "
        f"Train: {f1_train:.4f} | "
        f"Validación: {f1_val:.4f} | "
        f"Diferencia: {diferencia:.4f}")

Se evaluaron diferentes valores de min_samples_split para analizar su efecto sobre el rendimiento y el sobreajuste de Random Forest. Los valores 2, 5 y 10 produjeron resultados idénticos, debido a la restricción previamente establecida mediante min_samples_leaf=5.

Al aumentar min_samples_split a 20 y 50, se observó una reducción leve de la diferencia entre entrenamiento y validación, acompañada de una disminución del F1-score de validación.

Por lo tanto, se decidió mantener el valor predeterminado min_samples_split=2, ya que no se identificaron mejoras al modificar este parámetro.

# Evaluar max_features en Random Forest

In [ ]:
valores_max_features = ["sqrt", "log2", 0.3, 0.5, 0.8, 1.0]

resultados_features = []

for features in valores_max_features:

    modelo_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=5,
        min_samples_split=2,
        max_features=features,
        random_state=42,
        n_jobs=-1
    )

    modelo_rf.fit(X_entrenamiento, y_entrenamiento)

    pred_train = modelo_rf.predict(X_entrenamiento)
    pred_val = modelo_rf.predict(X_validacion)

    f1_train = f1_score(y_entrenamiento, pred_train)
    f1_val = f1_score(y_validacion, pred_val)

    diferencia = f1_train - f1_val

    resultados_features.append({
        "max_features": features,
        "F1 Train": f1_train,
        "F1 Validación": f1_val,
        "Diferencia": diferencia
    })

df_features = pd.DataFrame(resultados_features)

display(df_features.round(4))

## Regresión logística

# Evaluar regularización en Regresión Logística


In [ ]:

from sklearn.linear_model import LogisticRegression

valores_C = [0.001, 0.01, 0.1, 1, 10, 100]

resultados_logistica = []

for c in valores_C:

    modelo_logistico = LogisticRegression(
        C=c,
        max_iter=1000,
        random_state=42
    )

    modelo_logistico.fit(X_entrenamiento, y_entrenamiento)

    pred_train = modelo_logistico.predict(X_entrenamiento)
    pred_val = modelo_logistico.predict(X_validacion)

    f1_train = f1_score(y_entrenamiento, pred_train)
    f1_val = f1_score(y_validacion, pred_val)

    diferencia = f1_train - f1_val

    resultados_logistica.append({
        "C": c,
        "F1 Train": f1_train,
        "F1 Validación": f1_val,
        "Diferencia": diferencia
    })

df_logistica = pd.DataFrame(resultados_logistica)

display(df_logistica.round(4))

# Comparación de los tres modelos regularizados

In [ ]:


modelos_regularizados = {
    "Regresión Logística": LogisticRegression(
        C=0.1,
        max_iter=1000,
        random_state=42
    ),

    "Árbol de Decisión": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=5,
        min_samples_split=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )
}

resultados_finales = []

for nombre, modelo in modelos_regularizados.items():

    modelo.fit(X_entrenamiento, y_entrenamiento)

    pred_train = modelo.predict(X_entrenamiento)
    pred_val = modelo.predict(X_validacion)

    f1_train = f1_score(y_entrenamiento, pred_train)
    f1_val = f1_score(y_validacion, pred_val)

    resultados_finales.append({
        "Modelo": nombre,
        "F1 Train": f1_train,
        "F1 Validación": f1_val,
        "Diferencia": f1_train - f1_val
    })

df_comparacion = pd.DataFrame(resultados_finales)

display(df_comparacion.round(4))

# Comparación de F1-score entre los tres modelos


In [ ]:

df_comparacion.plot(
    x="Modelo",
    y=["F1 Train", "F1 Validación"],
    kind="bar",
    figsize=(9, 5)
)

plt.title("Comparación de modelos regularizados")
plt.xlabel("Modelo")
plt.ylabel("F1-score (fumadores)")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(["Entrenamiento", "Validación"])
plt.grid(axis="y")
plt.show()

## Configuración provisional y guardado

Este es el modelo elegido hasta el momento. Si se continúa la optimización con Train/Validación, actualizar esta sección **antes de volver a guardar y validar**.

In [ ]:

modelo_final = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=5,
    min_samples_split=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

modelo_final.fit(X_train, y_train)

print("Modelo Random Forest entrenado correctamente.")
print("Registros utilizados:", X_train.shape[0])
print("Características utilizadas:", X_train.shape[1])

# Guardar el modelo final en Google Drive


In [ ]:

import joblib
from pathlib import Path

ruta_modelos = Path("/content/drive/MyDrive/DiploIA/TP2/2-models")

# Crear la carpeta si todavía no existe
ruta_modelos.mkdir(parents=True, exist_ok=True)

# Definir el nombre del archivo
ruta_modelo_final = ruta_modelos / "random_forest_final.pkl"

# Guardar el modelo
joblib.dump(modelo_final, ruta_modelo_final)

print("Modelo guardado correctamente.")
print("Ubicación:", ruta_modelo_final)

## Próximo experimento: reducir profundidad

Se propone evaluar `max_depth` de 4 a 10, comparando F1 Train y Validación. **No se evalúa Test en esta etapa.**

In [ ]:
profundidades = [4, 5, 6, 7, 8, 9, 10]
resultados_profundidad = []

for profundidad in profundidades:
    modelo_rf = RandomForestClassifier(
        n_estimators=100, max_depth=profundidad, min_samples_leaf=5,
        min_samples_split=2, max_features="sqrt", random_state=42, n_jobs=-1
    )
    modelo_rf.fit(X_entrenamiento, y_entrenamiento)
    f1_train = f1_score(y_entrenamiento, modelo_rf.predict(X_entrenamiento))
    f1_val = f1_score(y_validacion, modelo_rf.predict(X_validacion))
    resultados_profundidad.append({
        "max_depth": profundidad, "F1 Train": f1_train,
        "F1 Validación": f1_val, "Diferencia": f1_train - f1_val
    })

display(pd.DataFrame(resultados_profundidad).round(4))